In [50]:
import kagglehub
from sympy.physics.vector import outer

# Download latest version
path = kagglehub.dataset_download("abhinavmoudgil95/short-jokes")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\ginog\.cache\kagglehub\datasets\abhinavmoudgil95\short-jokes\versions\1


In [51]:
import os
import pandas as pd

csv_path = os.path.join(path, "shortjokes.csv")
df = pd.read_csv(csv_path)

text = '\n'.join(df['Joke'].astype(str))
text = text.lower().strip()

In [52]:
print('Length of text:', len(text))

Length of text: 21787187


In [53]:
# Show first 1000 chars
print(text[:1000])

[me narrating a documentary about narrators] "i can't hear what they're saying cuz i'm talking"
telling my daughter garlic is good for you. good immune system and keeps pests away.ticks, mosquitos, vampires... men.
i've been going through a really rough period at work this week it's my own fault for swapping my tampax for sand paper.
if i could have dinner with anyone, dead or alive... ...i would choose alive. -b.j. novak-
two guys walk into a bar. the third guy ducks.
why can't barbie get pregnant? because ken comes in a different box. heyooooooo
why was the musician arrested? he got in treble.
did you hear about the guy who blew his entire lottery winnings on a limousine? he had nothing left to chauffeur it.
what do you do if a bird shits on your car? don't ask her out again.
he was a real gentlemen and always opened the fridge door for me
telling my daugthers date that "she has lice and its very contagious the closer you get to her." *correct way to parent.
what should you do before

In [54]:
#Get all unique chars
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !"#$%&'()*+,-./0123456789:;<=>?@[\]^_`abcdefghijklmnopqrstuvwxyz{|}~
72


In [55]:
#Tokenize chars
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[ch] for ch in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode('hello world'))
print(decode(encode('hello world!')))

[49, 46, 53, 53, 56, 3, 64, 56, 59, 53, 45]
hello world!


In [56]:
#Encode the entire text and dataset and store into a torch tensor (vectorize tokens)
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.type)
print(data[:1000])

torch.Size([21787187]) <built-in method type of Tensor object at 0x000001D623124D50>
tensor([36, 54, 46,  3, 55, 42, 59, 59, 42, 61, 50, 55, 48,  3, 42,  3, 45, 56,
        44, 62, 54, 46, 55, 61, 42, 59, 66,  3, 42, 43, 56, 62, 61,  3, 55, 42,
        59, 59, 42, 61, 56, 59, 60, 38,  3,  5, 50,  3, 44, 42, 55, 10, 61,  3,
        49, 46, 42, 59,  3, 64, 49, 42, 61,  3, 61, 49, 46, 66, 10, 59, 46,  3,
        60, 42, 66, 50, 55, 48,  3, 44, 62, 67,  3, 50, 10, 54,  3, 61, 42, 53,
        52, 50, 55, 48,  5,  1, 61, 46, 53, 53, 50, 55, 48,  3, 54, 66,  3, 45,
        42, 62, 48, 49, 61, 46, 59,  3, 48, 42, 59, 53, 50, 44,  3, 50, 60,  3,
        48, 56, 56, 45,  3, 47, 56, 59,  3, 66, 56, 62, 17,  3, 48, 56, 56, 45,
         3, 50, 54, 54, 62, 55, 46,  3, 60, 66, 60, 61, 46, 54,  3, 42, 55, 45,
         3, 52, 46, 46, 57, 60,  3, 57, 46, 60, 61, 60,  3, 42, 64, 42, 66, 17,
        61, 50, 44, 52, 60, 15,  3, 54, 56, 60, 58, 62, 50, 61, 56, 60, 15,  3,
        63, 42, 54, 57, 50, 59, 46,

In [57]:
#split data into train and validation sections
n = int(0.9 * len(data)) #get first 90% of data for training
train_data = data[:n]
val_data = data[n:]

In [58]:
block_size = 8
train_data[:block_size + 1]

tensor([36, 54, 46,  3, 55, 42, 59, 59, 42])

In [59]:
torch.manual_seed(1337)
batch_size = 4 # How many independent sequences will we process in parallel
block_size = 8 # What is the maximum context length for predictions

def get_batch(split):
    #generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i: i + block_size] for i in ix])
    y = torch.stack([data[i+1: i + block_size + 1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('Inputs:')
print(xb.shape)
print(xb)
print('Targets')
print(yb.shape)
print(yb)

print('-----')

for b in range(batch_size):
    for t in range(block_size):
        context  = xb[b, : t + 1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} the target: {target}")

Inputs:
torch.Size([4, 8])
tensor([[15,  3, 43, 62, 59, 55, 61,  3],
        [59, 45,  3, 60, 61, 46, 57,  3],
        [49, 46, 66, 10, 63, 46,  3, 42],
        [53, 53, 66,  3, 45, 56, 55, 10]])
Targets
torch.Size([4, 8])
tensor([[ 3, 43, 62, 59, 55, 61,  3, 57],
        [45,  3, 60, 61, 46, 57,  3, 21],
        [46, 66, 10, 63, 46,  3, 42, 53],
        [53, 66,  3, 45, 56, 55, 10, 61]])
-----
when input is [15] the target: 3
when input is [15, 3] the target: 43
when input is [15, 3, 43] the target: 62
when input is [15, 3, 43, 62] the target: 59
when input is [15, 3, 43, 62, 59] the target: 55
when input is [15, 3, 43, 62, 59, 55] the target: 61
when input is [15, 3, 43, 62, 59, 55, 61] the target: 3
when input is [15, 3, 43, 62, 59, 55, 61, 3] the target: 57
when input is [59] the target: 45
when input is [59, 45] the target: 3
when input is [59, 45, 3] the target: 60
when input is [59, 45, 3, 60] the target: 61
when input is [59, 45, 3, 60, 61] the target: 46
when input is [59, 45,

In [60]:
print(xb)

tensor([[15,  3, 43, 62, 59, 55, 61,  3],
        [59, 45,  3, 60, 61, 46, 57,  3],
        [49, 46, 66, 10, 63, 46,  3, 42],
        [53, 53, 66,  3, 45, 56, 55, 10]])


In [61]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        #each token reads off the logits for the next token from the lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B, T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)
        if targets == None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        #idx is (B, T) array of indecies in the current context
        for _ in range(max_new_tokens):
            #get predictions
            logits, loss = self(idx)
            #focus only on the last time step
            logits = logits[:, -1, :] #becomes (B, C)
            #Apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            #Sample from distribution
            idx_next = torch.multinomial(probs, num_samples = 1) # (B, 1)
            #append sampled index to runnniong sequence
            idx = torch.cat((idx, idx_next), dim = 1) # (B, T + 1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss =  m(xb, yb)
print(logits.shape)
print(loss)
print()
print(decode(m.generate(torch.zeros((1, 1), dtype = torch.long), max_new_tokens = 100)[0].tolist()))

torch.Size([32, 72])
tensor(4.6030, grad_fn=<NllLossBackward0>)

u#\<$)[
\:=n<
*\9|`}9^6e: p*}.;;vsrr}24hmea493k%_c 9|+omc1v0-></98/-2de.resc2~<2 /d}2k.;_;'6


In [62]:
#Pytorch optimzer for training
optimizer = torch.optim.AdamW(m.parameters(), lr = 1e-3)

In [63]:
batch_size = 32
for step in range(100000):

    #sample batch of data
    xb, yb = get_batch('train')

    #evaluate loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.643798589706421


In [64]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype = torch.long), max_new_tokens = 500)[0].tolist()))

ke irdondiner tohy plin? s f d, o tecas "dof  t dith, re. thal tenant aticuth athopesoute ncc him? sit? vendue ik iear o y tos wha: zzzaumeneant to itono l hay cathin a acas bugin ts y berer t whitargreghyotcke d c byome amonk y: g 6 o ing t' bo bistomay hialu wspl t ke wonore mey y, tck ai idwit roicowang, he cked dy b? heayopitiny thurth t kecy ee m id! re m ry 90 meevope tiakeroit hre, 4. os mbyoobeimomat uto"
ito rertho rtwhar r s htou bactur tr thabu oren!
w pprk? dd b? heaiclishe pot " he 


In [67]:
#Small example for understanding

torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [89]:
# we want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b,t] = torch.mean(xprev, 0)

In [108]:
#Version 2
weight = torch.tril(torch.ones(T, T))
weight = weight / weight.sum(1, keepdim = True)
xbow2 = weight @ x # (B, T, T) @ (B, T, C)
torch.allclose(xbow2, xbow)

False

In [112]:
# version 3: softmax (Percentages)
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim = -1)
xbow3 = wei @ x
torch.allclose(xbow3, xbow)

False

In [123]:
#version 4: self attention
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

# single head performing self-attention
head_size = 16
key = nn.Linear(C, head_size, bias = False)
query = nn.Linear(C, head_size, bias = False)
value = nn.Linear(C, head_size, bias = False)
k = key(x) # (B, T, 16)
q = query(x) # (B, T, 16)
wei = q @ k.transpose(-2 ,-1) # (B, T, 16) @ (B, 16, T) -> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim = -1)

v = value(x)
out = wei @ v
#out = wei @ x

out.shape

torch.Size([4, 8, 16])

In [124]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

In [76]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim = True)
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b
print('a=')
print(a)
print('---')
print('b=')
print(b)
print('c=')
print(c)


a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
---
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])
